In [1]:
import sys
print(sys.executable)

c:\Users\Lenovo\Desktop\Email-assistant\.venv\Scripts\python.exe


In [2]:
import sys
!{sys.executable} -m pip install -q \
    langchain-community \
    langchain-openai \
    langchain \
    langgraph \
    langgraph-prebuilt \
    python-dotenv \
    google-auth-oauthlib \
    google-auth-httplib2 \
    googleapis-common-protos \
    google-api-python-client


!{sys.executable} -m pip install -q langchain-groq
!{sys.executable} -m pip install -q beautifulsoup4
!{sys.executable} -m pip install -q langchain-google-genai

In [ ]:
import os
import email
import base64
import traceback
import os
import ssl
import certifi
import time
from datetime import datetime


# ✅ Fix SSL FIRST before any other imports
ssl._create_default_https_context = ssl.create_default_context
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

import httplib2
httplib2.Http(ca_certs=certifi.where())

# Patch httplib2 globally
import functools
original_http = httplib2.Http.__init__
def patched_http_init(self, *args, **kwargs):
    kwargs.setdefault('ca_certs', certifi.where())
    original_http(self, *args, **kwargs)
httplib2.Http.__init__ = patched_http_init



from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("GROQ_API_KEY")
#if not api_key:
#    raise ValueError("The GOOGLE_API_KEY environment variable is not set")
#print(api_key[:20])

# ── Gmail setup ───────────────────────────────────────────────────────
from langchain_community.agent_toolkits import GmailToolkit
from langchain_community.tools.gmail.utils import build_resource_service, get_gmail_credentials
from langchain_community.tools.gmail.search import GmailSearch, Resource
from langchain_community.tools.gmail.get_message import GmailGetMessage


In [4]:
credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://mail.google.com/"],
    client_secrets_file="credentials.json",
)
api_resource = build_resource_service(credentials=credentials)
toolkit = GmailToolkit(api_resource=api_resource)
original_tools = toolkit.get_tools()

In [5]:
from pydantic import BaseModel, Field, field_validator
from langchain_community.tools.gmail.search import Resource

class FixedSearchSchema(BaseModel):
    query: str = Field(description="Gmail search query")
    max_results: int = Field(default=10, description="Max number of results")
    resource: str = Field(default="messages", description="messages or threads")

    @field_validator("resource", mode="before")
    @classmethod
    def fix_resource(cls, v):
        if isinstance(v, Resource):
            return v.value
        return v

class FixedGmailSearch(GmailSearch):
    args_schema: type = FixedSearchSchema

    def _run(self, query: str, max_results: int = 10, resource: str = "messages"):
        if isinstance(resource, str):
            resource = Resource(resource)
        return super()._run(query=query, max_results=max_results, resource=resource)

In [12]:
def _fixed_parse_messages(self, results):
    messages = []

    if isinstance(results, list):
        items = results
    elif isinstance(results, dict):
        items = results.get("messages", results.get("threads", []))
    else:
        return str(results)

    for res in items[:10]:
        msg_id = res["id"] if isinstance(res, dict) else res
        message = self.api_resource.users().messages().get(
             userId="me",
            id=msg_id,
            format="metadata",
            fields="payload/headers"
         ).execute()

        headers = {
            h["name"].lower(): h["value"]
            for h in message.get("payload", {}).get("headers", [])
        }

        messages.append({
            "subject": headers.get("subject", "(no subject)"),
            "from":    headers.get("from", ""),
        })

    return str(messages)

GmailSearch._parse_messages = _fixed_parse_messages

In [13]:
class FixedGmailGetMessage(GmailGetMessage):
    def _run(self, message_id: str):
        try:
            message = self.api_resource.users().messages().get(
                userId="me",
                id=message_id,
                format="metadata",
                fields="payload/headers"
            ).execute()

            headers = {
                h["name"].lower(): h["value"]
                for h in message.get("payload", {}).get("headers", [])
            }

            return {
                "subject": headers.get("subject", "(no subject)"),
                "from":    headers.get("from", ""),
            }

        except Exception as e:
            return f"Error: {str(e)}"

In [14]:
fixed_tools = [
    FixedGmailSearch(api_resource=api_resource),
    FixedGmailGetMessage(api_resource=api_resource),
]

In [15]:
# ── LLM + Agent ───────────────────────────────────────────────────────
from langchain.agents import create_agent

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

today = datetime.now().strftime("%Y/%m/%d")

instructions = f"""Email assistant. Today = {today}.
- Search emails: use query strings like after:{today} or in:inbox
- For each result return: subject and sender only
- max_results must be an integer
- Count list items to answer "how many" questions"""

agent = create_agent(
    model=llm,
    tools=fixed_tools,
    system_prompt=instructions
)

In [16]:
import logging
logging.basicConfig(level=logging.WARNING)  # suppress info noise

import time

def run_agent(query: str, retries: int = 3):
    for attempt in range(retries):
        try:
            print(f"🔄 Attempt {attempt + 1}: {query}")
            result = agent.invoke({
                "messages": [{"role": "user", "content": query}]
            })
            response = result["messages"][-1].content
            print(f"✅ Done: {response}")
            return response
        except Exception as e:
            error_name = type(e).__name__
            error_msg = str(e)[:200]
            print(f"❌ Error type : {error_name}")
            print(f"❌ Error message: {error_msg}")
            if "SSL" in error_name or "SSL" in error_msg:
                print(f"⚠️ SSL error — retrying in 3 seconds...")
                time.sleep(3)
            else:
                import traceback
                traceback.print_exc()
                break
    print("❌ All retries failed")

run_agent(f"Search emails after {today} and tell me how many I received today and list their subjects")

🔄 Attempt 1: Search emails after 2026/05/08 and tell me how many I received today and list their subjects
✅ Done: You received 3 emails today. The subjects of the emails are:
1. Un poste comme IAM Architect / Wallix chez Visian et 3\xa0autres emplois pour France vous attendent. Postulez maintenant.
2. INTERNSHIP - AI SOFTWARE ENGINEER chez Trekea Mobile
3. How to Become Ridiculously More Creative | Viktoria Verde, PhD in Write A Catalyst


'You received 3 emails today. The subjects of the emails are:\n1. Un poste comme IAM Architect / Wallix chez Visian et 3\\xa0autres emplois pour France vous attendent. Postulez maintenant.\n2. INTERNSHIP - AI SOFTWARE ENGINEER chez Trekea Mobile\n3. How to Become Ridiculously More Creative | Viktoria Verde, PhD in Write A Catalyst'